In [1]:
from google.colab import files
files.upload()

Saving business.csv to business.csv
Saving economy.csv to economy.csv


<uploaded file contents omitted to reduce notebook size>

In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, root_mean_squared_error
import warnings
warnings.filterwarnings('ignore')

import os

import tensorflow
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [3]:
os.listdir()

['.config', 'economy.csv', 'business.csv', 'sample_data']

In [4]:
df_economy = pd.read_csv('economy.csv')

In [5]:
df_business = pd.read_csv('business.csv')

In [6]:
df_economy['class'] = 'economy'
df_business['class'] = 'business'

In [7]:
df = pd.concat([df_business, df_economy],axis=0)

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 300261 entries, 0 to 206773
Data columns (total 12 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   date        300261 non-null  object
 1   airline     300261 non-null  object
 2   ch_code     300261 non-null  object
 3   num_code    300261 non-null  int64 
 4   dep_time    300261 non-null  object
 5   from        300261 non-null  object
 6   time_taken  300261 non-null  object
 7   stop        300261 non-null  object
 8   arr_time    300261 non-null  object
 9   to          300261 non-null  object
 10  price       300261 non-null  object
 11  class       300261 non-null  object
dtypes: int64(1), object(11)
memory usage: 29.8+ MB


In [9]:
df.head()

,date,airline,ch_code,num_code,dep_time,from,time_taken,stop,arr_time,to,price,class
0,11-02-2022,Air India,AI,868,18:00,Delhi,02h 00m,non-stop,20:00,Mumbai,"25,612",business
1,11-02-2022,Air India,AI,624,19:00,Delhi,02h 15m,non-stop,21:15,Mumbai,"25,612",business
2,11-02-2022,Air India,AI,531,20:00,Delhi,24h 45m,1-stop\n\t\t\t\t\t\t\t\t\t\t\t\t\n\t\t\t\t\t\t...,20:45,Mumbai,"42,220",business
3,11-02-2022,Air India,AI,839,21:25,Delhi,26h 30m,1-stop\n\t\t\t\t\t\t\t\t\t\t\t\t\n\t\t\t\t\t\t...,23:55,Mumbai,"44,450",business
4,11-02-2022,Air India,AI,544,17:15,Delhi,06h 40m,1-stop\n\t\t\t\t\t\t\t\t\t\t\t\t\n\t\t\t\t\t\t...,23:55,Mumbai,"46,690",business


In [10]:
df['date'] = pd.to_datetime(df['date'],format='%d-%m-%Y')

In [11]:
df['dep_time'] = pd.to_datetime(df['dep_time'],format='%H:%M')

In [12]:
df.head()

,date,airline,ch_code,num_code,dep_time,from,time_taken,stop,arr_time,to,price,class
0,2022-02-11,Air India,AI,868,1900-01-01 18:00:00,Delhi,02h 00m,non-stop,20:00,Mumbai,"25,612",business
1,2022-02-11,Air India,AI,624,1900-01-01 19:00:00,Delhi,02h 15m,non-stop,21:15,Mumbai,"25,612",business
2,2022-02-11,Air India,AI,531,1900-01-01 20:00:00,Delhi,24h 45m,1-stop\n\t\t\t\t\t\t\t\t\t\t\t\t\n\t\t\t\t\t\t...,20:45,Mumbai,"42,220",business
3,2022-02-11,Air India,AI,839,1900-01-01 21:25:00,Delhi,26h 30m,1-stop\n\t\t\t\t\t\t\t\t\t\t\t\t\n\t\t\t\t\t\t...,23:55,Mumbai,"44,450",business
4,2022-02-11,Air India,AI,544,1900-01-01 17:15:00,Delhi,06h 40m,1-stop\n\t\t\t\t\t\t\t\t\t\t\t\t\n\t\t\t\t\t\t...,23:55,Mumbai,"46,690",business


In [14]:
df['price'] = df['price'].str.replace('₹', '').str.replace(',', '').astype('int')

In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 300261 entries, 0 to 206773
Data columns (total 12 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   date        300261 non-null  datetime64[ns]
 1   airline     300261 non-null  object        
 2   ch_code     300261 non-null  object        
 3   num_code    300261 non-null  int64         
 4   dep_time    300261 non-null  datetime64[ns]
 5   from        300261 non-null  object        
 6   time_taken  300261 non-null  object        
 7   stop        300261 non-null  object        
 8   arr_time    300261 non-null  object        
 9   to          300261 non-null  object        
 10  price       300261 non-null  int64         
 11  class       300261 non-null  object        
dtypes: datetime64[ns](2), int64(2), object(8)
memory usage: 29.8+ MB


In [16]:
df = pd.get_dummies(df,columns=['airline','ch_code','from','stop','to','class'],drop_first=True)

In [17]:
df.columns = df.columns.str.replace(' ','_')

In [18]:
df_new = df.drop(['date','dep_time','time_taken','arr_time'],axis=1)

In [60]:
df_numeric_for_vif = df_new.astype(float)

VIF = pd.DataFrame()
VIF['variables'] = df_numeric_for_vif.columns
VIF['VIF'] = [variance_inflation_factor(df_numeric_for_vif.values,i) for i in range(df_numeric_for_vif.shape[1])]
VIF

,variables,VIF
0,num_code,1.384991
1,price,10.518536
2,airline_AirAsia,inf
3,airline_GO_FIRST,inf
4,airline_Indigo,inf
...,...,...
61,to_Delhi,1.876746
62,to_Hyderabad,1.649548
63,to_Kolkata,1.735936
64,to_Mumbai,1.858227


In [70]:
VIF.loc[(VIF['VIF'] > 1.5) & (VIF['VIF'] != np.inf),'variables'].values

array(['price', 'ch_code_AI', 'from_Chennai', 'from_Delhi',
       'from_Hyderabad', 'from_Kolkata', 'from_Mumbai', 'to_Chennai',
       'to_Delhi', 'to_Hyderabad', 'to_Kolkata', 'to_Mumbai',
       'class_economy'], dtype=object)

In [19]:
df_filter = df_new[['price', 'ch_code_AI', 'from_Chennai', 'from_Delhi',
       'from_Hyderabad', 'from_Kolkata', 'from_Mumbai', 'to_Chennai',
       'to_Delhi', 'to_Hyderabad', 'to_Kolkata', 'to_Mumbai',
       'class_economy']]

In [20]:
x = df_filter.drop('price',axis=1).values
y = df_filter['price'].values

In [21]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)

In [22]:
std = StandardScaler()

In [23]:
x_train_std = std.fit_transform(x_train)
x_test_std = std.transform(x_test)

In [24]:
model = Sequential()

In [105]:
Dense?

In [26]:
model.add(Dense(units=50,activation='relu',input_dim=x_train_std.shape[1]))
model.add(Dropout(0.2))
model.add(Dense(units=30,activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(units=1,activation='sigmoid'))

In [28]:
model.compile?

In [27]:
model.compile(optimizer='adam',loss='mse',metrics=['mae'])

In [29]:
model.fit(x_train_std,y_train,epochs=10,batch_size=32,validation_split=0.2)

Epoch 1/10
6006/6006 ━━━━━━━━━━━━━━━━━━━━ 21s 3ms/step - loss: 952614912.0000 - mae: 20894.0664 - val_loss: 949430400.0000 - val_mae: 20894.9414
Epoch 2/10
6006/6006 ━━━━━━━━━━━━━━━━━━━━ 18s 3ms/step - loss: 952615232.0000 - mae: 20894.0996 - val_loss: 949430400.0000 - val_mae: 20894.9414
Epoch 3/10
6006/6006 ━━━━━━━━━━━━━━━━━━━━ 18s 3ms/step - loss: 952614912.0000 - mae: 20894.0723 - val_loss: 949430400.0000 - val_mae: 20894.9414
Epoch 4/10
6006/6006 ━━━━━━━━━━━━━━━━━━━━ 20s 3ms/step - loss: 952616896.0000 - mae: 20894.1074 - val_loss: 949430400.0000 - val_mae: 20894.9414
Epoch 5/10
6006/6006 ━━━━━━━━━━━━━━━━━━━━ 18s 3ms/step - loss: 952616640.0000 - mae: 20894.0469 - val_loss: 949430400.0000 - val_mae: 20894.9414
Epoch 6/10
6006/6006 ━━━━━━━━━━━━━━━━━━━━ 18s 3ms/step - loss: 952615808.0000 - mae: 20894.0898 - val_loss: 949430400.0000 - val_mae: 20894.9414
Epoch 7/10
6006/6006 ━━━━━━━━━━━━━━━━━━━━ 20s 3ms/step - loss: 952616320.0000 - mae: 20894.0664 - val_loss: 949430400.0000 - val_m

In [30]:
y_pred_train = (model.predict(x_train_std)>0.5)
y_pred_test = (model.predict(x_test_std)>0.5)

7507/7507 ━━━━━━━━━━━━━━━━━━━━ 10s 1ms/step
1877/1877 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step


In [31]:
root_mean_squared_error(y_train,y_pred_train)

np.float64(30854.15207940125)

In [32]:
root_mean_squared_error(y_test,y_pred_test)

np.float64(30790.227333433802)